# TER Picopatt - Clustering

Lecture et préparation des données

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "src").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import time
import glob
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    pairwise_distances,
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score,
    adjusted_rand_score
)

import picopatt as fc
import contextily as ctx

# Lecture de tes fichiers CSV
# Dossier contenant tous les fichiers "clean_nozeros"
DATA_NOZERO = Path("data/processed/picopatt/clean_nozeros")
bd = fc.load_all(DATA_NOZERO, False)

print(f"Données chargées : {len(bd)} points")
print(f"Variables disponibles : {list(bd.columns)}")

In [ ]:
# Sélection des variables à utiliser pour le clustering
meteo_cols = [col for col in bd.columns if any(key in col for key in ["tair", "rh", "sw", "lw", "tmrt", "pet"])]

# Garder seulement les lignes sans valeurs manquantes sur ces variables
bd = bd[meteo_cols + ["lon_ontrack", "lat_ontrack", "timestamp"]].dropna()
print(f"Données prêtes : {len(bd)} points, {len(meteo_cols)} variables")

In [ ]:
# Standardisation (centrer-réduire)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(bd[meteo_cols])
print("Données standardisées (centrées-réduites)")

In [ ]:
# Clustering K-Means sur les points de mesure
# Objectif :
# Regrouper les points de mesure similaires en fonction de leurs caractéristiques physiques :
# - température de surface (tair_tc1, tair_tc2)
# - humidité relative (rh_thermohygro)
# - vitesse du vent (ws)
# - rayonnement solaire réfléchi (sw_up)
# - rayonnement infrarouge descendant (lw_down)
#
# Le clustering est effectué sur les points (et non les moyennes par parcours).
# Chaque ligne du DataFrame représente un point géographique mesuré dans la ville.
# Après standardisation des variables, le modèle K-Means sépare les points
# en 4 groupes (clusters) de comportements microclimatiques similaires.


# Nombre de clusters à créer
nb_clusters = 3

#  Initialisation du modèle K-Means
kmeans = KMeans(
    n_clusters=nb_clusters,  # nombre de clusters
    random_state=42,         # pour garantir la reproductibilité
    n_init=10                # nombre d'initialisations pour stabilité
)

# Application du clustering sur les données standardisées
labels = kmeans.fit_predict(X_scaled)

# Ajout du label de cluster dans le DataFrame d'origine
bd["cluster"] = labels

# Affichage du résultat global
print("\n Clustering terminé.")
print("Répartition du nombre de points par cluster :")
print(bd["cluster"].value_counts())

Interprétation :

Le modèle K-Means a formé 4 groupes de points présentant des conditions microclimatiques distinctes. Chaque cluster regroupe des points ayant des valeurs similaires de température, humidité, vent et rayonnement. Ces chiffres indiquent combien de points appartiennent à chaque cluster.
Par exemple, ici :
- Le cluster 3 est le plus fréquent (~88 000 points)
- Le cluster 2 est le plus rare (~17 000 points)

Cela montre la diversité des situations microclimatiques observées sur l’ensemble des points mesurés.

Le clustering K-Means regroupe les points de mesure selon leur similitude physique :
- Chaque point est défini par ses 6 variables (température, humidité, vent, rayonnements).
- Chaque cluster correspond à un type de microclimat observé sur le terrain.
- Le nombre de points dans chaque cluster reflète la fréquence de ce type de conditions.

En résumé :
- Le cluster 3 est le plus représenté (conditions les plus fréquentes).
- Le cluster 2 est minoritaire, indiquant un type de situation plus rare (ex. fortes chaleurs ou conditions extrêmes).
- Ces résultats serviront à identifier les signatures microclimatiques typiques avant l’intégration des vecteurs AlphaEarth.

In [ ]:
centroids = kmeans.cluster_centers_

# Liste pour stocker les indices des médoides
indexes_medoids = []

# Boucle sur chaque cluster
for k in range(nb_clusters):
    # Sélectionner les points appartenant au cluster k
    cluster_points = X_scaled[bd["cluster"] == k]
    
    # Calculer la distance de chaque point au centroide du cluster
    distances = pairwise_distances(centroids[k].reshape(1, -1), cluster_points)
    
    # Trouver l'indice du point le plus proche du centroide
    idx_min = np.argmin(distances)
    
    # Récupérer l'indice global dans le DataFrame original
    idx_global = bd[bd["cluster"] == k].index[idx_min]

        # Ajouter cet indice à la liste des médoides
    indexes_medoids.append(idx_global)

# Extraire les points médoides du DataFrame
medoids = bd.loc[indexes_medoids]

print("\nMédoides calculés :")
display(medoids[meteo_cols + ["lon_ontrack", "lat_ontrack"]])

Interprétation

Les médoides permettent de représenter chaque cluster par un point réel du jeu de données. Contrairement aux centroides, ils ne sont pas théoriques : ils correspondent à des mesures réelles dans la ville.

Chaque médoide illustre les conditions microclimatiques typiques de son groupe :
- Cluster 0 -> chaud et humide
- Cluster 1 -> froid et humide
- Cluster 2 -> très chaud et lumineux
- Cluster 3 -> modéré et ventilé

Ces points sont utiles pour visualiser les signatures microclimatiques et pour interpréter le comportement des zones dans le temps ou l’espace.

In [ ]:
# Moyenne des variables par cluster

# Objectif :
# Calculer la moyenne des variables physiques (température, humidité, vent, rayonnement)
# pour chaque cluster afin de caractériser les différents types de microclimats détectés.

# Calcul de la moyenne des indicateurs par cluster
mean_per_cluster = bd.groupby("cluster")[meteo_cols].mean().round(2)

print("\n Moyenne des variables par cluster :")
display(mean_per_cluster)

 Interprétation des moyennes de clusters

Cette étape permet d’identifier les types de microclimats présents dans les données. Chaque cluster regroupe des points de mesure aux caractéristiques similaires :
- Cluster 0 : chaud et humide, typique de zones urbaines confinées ou très exposées.
- Cluster 1 : froid et humide, correspondant à des zones ombragées ou matinales.
- Cluster 2 : très chaud, fort rayonnement (îlot de chaleur ou surfaces minérales exposées).
- Cluster 3 : modéré, sec et plus ventilé (zones ouvertes, proches d’axes aérés).

Ces profils moyens permettront de comparer les signatures microclimatiques entre zones, ou d’observer comment elles évoluent avec l’ajout des vecteurs AlphaEarth.

In [ ]:
# Visualisation spatiale des clusters

# Objectif :
# Représenter la position géographique des points (lon_rtk / lat_rtk)
# colorés selon leur cluster pour observer les zones où les conditions
# microclimatiques sont similaires.

CLUST = Path("analyse_exploratoire/figures/clusters")
fc.create_folder(CLUST)

plt.figure(figsize=(10, 8)) # Un peu plus grand pour la lisibilité

# 1. Tracer vos points
# Note : On utilise zorder pour s'assurer que les points sont au-dessus de la carte
scatter = plt.scatter(
    bd["lon_ontrack"], bd["lat_ontrack"],
    c=bd["cluster"], cmap="tab10", s=5, alpha=0.7, zorder=2
)

# 2. Tracer les médoïdes
plt.scatter(
    medoids["lon_ontrack"], medoids["lat_ontrack"],
    c="black", s=100, marker="X", label="Médoïdes", zorder=3
)

# 3. Ajouter le fond de carte
# crs="EPSG:4326" indique à contextily que vos données sont en Latitude/Longitude
ctx.add_basemap(plt.gca(), crs="EPSG:4326", source=ctx.providers.OpenStreetMap.Mapnik)

plt.title("Répartition spatiale des clusters - Montpellier")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.legend()
plt.tight_layout()

plt.savefig(CLUST / "repart_spatial_clusters_map.png", dpi=150)
plt.show()

Interprétation à ajouter dans ton rapport

Répartition spatiale des clusters

Cette carte représente les points de mesure GPS (lon_rtk, lat_rtk) colorés selon leur cluster microclimatique :
- Chaque couleur correspond à un groupe de points partageant des caractéristiques thermiques et radiatives similaires.
- On observe que certaines zones de la ville sont dominées par un cluster précis, indiquant un type de microclimat local.

Par exemple :
- une zone bleue majoritaire, conditions fraîches et ombragées,
- une zone rouge, forte exposition solaire ou chaleur plus élevée,
- une zone verte, conditions modérées et ventilées.

Ces cartes permettent de visualiser la distribution spatiale des microclimats urbains et de repérer les zones à fort contraste thermique.

In [ ]:
# Export du fichier CSV pour Google Earth Engine

# On garde uniquement les colonnes utiles pour la visualisation
cols_to_export = ["lon_ontrack", "lat_ontrack", "cluster"]

# Vérifier que les colonnes existent
print("Colonnes présentes :", [c for c in cols_to_export if c in bd.columns])

# Créer un DataFrame pour l’export
bd_export = bd[cols_to_export].dropna()

# Enregistrer le fichier CSV 
AED = Path("data/processed/alphaearth/alphaearth_data")
fc.create_folder(AED)

output_path = AED / "points_clusters.csv"
bd_export.to_csv(output_path, index=False)

print(f" Fichier exporté : {output_path}")
print(f" Nombre de points exportés : {len(bd_export)}")

Utilisation des données AlphaEarth issues du notebook Alphaearth.ipynb

In [ ]:
# Adapter bd pour integrer les AE
# Standardiser lon/lat
bd = bd.copy()
bd["lon_std"] = pd.to_numeric(bd["lon_ontrack"], errors="coerce")
bd["lat_std"] = pd.to_numeric(bd["lat_ontrack"], errors="coerce")

bd = bd.dropna(subset=["lon_std", "lat_std"])
bd = bd[(bd["lon_std"].between(-180, 180)) & (bd["lat_std"].between(-90, 90))]

# UID stable = index après reset
bd = bd.reset_index(drop=True)
bd["uid"] = bd.index.astype(int)

print("bd shape:", bd.shape)
print(bd[["uid","lon_std","lat_std"]].head())

In [ ]:
AE = Path("data/processed/alphaearth/alphaearth_data")

# Fichiers exports AlphaEarth
raw_path  = AE / "alphaearth_A00_A63_points.csv"
pca10_path = AE / "alphaearth_pca10_points.csv"
pca15_path = AE / "alphaearth_pca15_points.csv"
pca32_path = AE / "alphaearth_pca32_points.csv"

for p in [raw_path, pca10_path, pca15_path, pca32_path]:
    if not p.exists():
        raise FileNotFoundError(f"Fichier manquant: {p}")

aef_raw  = pd.read_csv(raw_path)
aef_pca10 = pd.read_csv(pca10_path)
aef_pca15 = pd.read_csv(pca15_path)
aef_pca32 = pd.read_csv(pca32_path)

# Noms des colonnes
BANDS = [f"A{i:02d}" for i in range(64)]
PCA10_COLS = [c for c in aef_pca10.columns if c.startswith("aef_pca10_")]
PCA15_COLS = [c for c in aef_pca15.columns if c.startswith("aef_pca15_")]
PCA32_COLS = [c for c in aef_pca32.columns if c.startswith("aef_pca32_")]

# Merge dans bd
df = bd.merge(aef_raw[["uid"] + BANDS], on="uid", how="left")
df = df.merge(aef_pca10[["uid"] + PCA10_COLS], on="uid", how="left")
df = df.merge(aef_pca15[["uid"] + PCA15_COLS], on="uid", how="left")
df = df.merge(aef_pca32[["uid"] + PCA32_COLS], on="uid", how="left")

print("df merged shape:", df.shape)
print("NaN embeddings raw A00:", int(df["A00"].isna().sum()))
print("NaN PCA10:", int(df[PCA10_COLS].isna().any(axis=1).sum()))

In [ ]:
# Batch tests KMeans Picopatt
# k = 3..6, sauvegarde figures + CSV résultats par k

# metrics

def silhouette_stratified(X_scaled, labels, per_cluster=3000, random_state=42):
    """
    Silhouette sur échantillon stratifié par cluster (plus stable que random).
    """
    rng = np.random.default_rng(random_state)
    labels = np.asarray(labels)

    idx_all = []
    for kk in np.unique(labels):
        idx_k = np.where(labels == kk)[0]
        if len(idx_k) == 0:
            continue
        m = min(per_cluster, len(idx_k))
        idx_all.append(rng.choice(idx_k, size=m, replace=False))

    idx = np.concatenate(idx_all)
    return float(silhouette_score(X_scaled[idx], labels[idx]))


def silhouette_bootstrap(X_scaled, labels, per_cluster=2000, n_boot=10, random_state=42):
    """
    Bootstrap silhouette stratifié -> moyenne + écart-type.
    """
    rng = np.random.default_rng(random_state)
    scores = []
    for _ in range(n_boot):
        rs = int(rng.integers(0, 1_000_000))
        scores.append(silhouette_stratified(X_scaled, labels, per_cluster=per_cluster, random_state=rs))
    scores = np.asarray(scores)
    return float(scores.mean()), float(scores.std())


def kmeans_stability_ari(X_scaled, k, n_runs=3, base_seed=42, n_init=10):
    """
    Stabilité des clusters : ARI pair-à-pair entre plusieurs runs KMeans.
    """
    labels_list = []
    for i in range(n_runs):
        rs = base_seed + i
        km = KMeans(n_clusters=k, random_state=rs, n_init=n_init)
        labels_list.append(km.fit_predict(X_scaled))

    aris = []
    for i in range(n_runs):
        for j in range(i + 1, n_runs):
            aris.append(adjusted_rand_score(labels_list[i], labels_list[j]))

    aris = np.asarray(aris)
    return float(aris.mean()), float(aris.std())


In [ ]:
# Function: fit + medoids + plot(save only) + metrics

def kmeans_clusters_medoids_save(
    df_in: pd.DataFrame,
    feat_cols: list,
    name: str,
    k: int,
    out_dir_fig: Path,
    out_dir_medoids: Path,
    lon_col: str = "lon_ontrack",
    lat_col: str = "lat_ontrack",
    sample_sil_per_cluster: int = 3000,
    boot_per_cluster: int = 2000,
    boot_n: int = 10,
    stability_runs: int = 3,
    plot_max: int = 120000,
    random_state: int = 42,
    n_init: int = 10
):
    """
    - dropna sur coords + features
    - StandardScaler
    - KMeans
    - Médoïdes (point réel le plus proche du centroïde)
    - Plot spatial sauvegardé uniquement
    - Metrics: silhouette stratifié + bootstrap, DB, CH, inertia, stabilité ARI, tailles clusters
    - Sauvegarde médoïdes en CSV
    """
    t0 = time.time()

    needed = [lon_col, lat_col] + feat_cols
    d = df_in[needed].dropna().copy()
    n = len(d)
    if n < 1000:
        print(f"[SKIP] {name} k={k} -> trop peu de lignes après dropna: n={n}")
        return None

    # Standardisation
    t1 = time.time()
    scaler = StandardScaler()
    X = d[feat_cols].to_numpy()
    X_scaled = scaler.fit_transform(X)
    t_scale = time.time() - t1

    # KMeans fit
    t2 = time.time()
    km = KMeans(n_clusters=k, random_state=random_state, n_init=n_init)
    labels = km.fit_predict(X_scaled)
    inertia = float(km.inertia_)
    t_fit = time.time() - t2
    d["cluster"] = labels

    # Metrics (rapides)
    t3 = time.time()

    # Silhouette stratifié + bootstrap
    sil_strat = np.nan
    sil_mean, sil_std = (np.nan, np.nan)
    if k > 1:
        sil_strat = silhouette_stratified(
            X_scaled, labels,
            per_cluster=sample_sil_per_cluster,
            random_state=random_state
        )
        sil_mean, sil_std = silhouette_bootstrap(
            X_scaled, labels,
            per_cluster=boot_per_cluster,
            n_boot=boot_n,
            random_state=random_state
        )

    # Davies-Bouldin / Calinski-Harabasz (calculés sur tout X_scaled)
    db = float(davies_bouldin_score(X_scaled, labels)) if k > 1 else np.nan
    ch = float(calinski_harabasz_score(X_scaled, labels)) if k > 1 else np.nan

    # Stabilité ARI (refait des KMeans) -> coûte du temps, mais utile
    ari_mean, ari_std = (np.nan, np.nan)
    if k > 1 and stability_runs >= 2:
        ari_mean, ari_std = kmeans_stability_ari(
            X_scaled, k,
            n_runs=stability_runs,
            base_seed=random_state,
            n_init=n_init
        )

    # Distribution clusters
    counts = pd.Series(labels).value_counts().sort_index()
    min_c = int(counts.min())
    max_c = int(counts.max())
    ratio_max_min = float(max_c / max(min_c, 1))

    t_metrics = time.time() - t3

    # Médoïdes
    t4 = time.time()
    centroids = km.cluster_centers_
    indexes_medoids = []
    for kk in range(k):
        mask = (labels == kk)
        cluster_points = X_scaled[mask]
        # si cluster vide (rare) -> skip
        if cluster_points.shape[0] == 0:
            indexes_medoids.append(None)
            continue
        dist = pairwise_distances(centroids[kk].reshape(1, -1), cluster_points)
        idx_min = int(np.argmin(dist))
        idx_global = d[mask].index[idx_min]
        indexes_medoids.append(idx_global)

    medoids = d.loc[[ix for ix in indexes_medoids if ix is not None], [lon_col, lat_col, "cluster"] + feat_cols].copy()
    medoids_path = out_dir_medoids / f"medoids_{name}_k{k}.csv"
    medoids.to_csv(medoids_path, index=False)
    t_med = time.time() - t4

    # Plot (save only)
    t5 = time.time()
    if plot_max is not None and n > plot_max:
        rng = np.random.default_rng(random_state)
        keep = rng.choice(n, size=plot_max, replace=False)
        dp = d.iloc[keep].copy()
    else:
        dp = d

    fig_path = out_dir_fig / f"clusters_{name}_k{k}.png"
    plt.figure(figsize=(8, 6))
    plt.scatter(dp[lon_col], dp[lat_col], c=dp["cluster"], cmap="tab10", s=2, alpha=0.6)
    plt.scatter(medoids[lon_col], medoids[lat_col], c="black", s=80, marker="X", label="Médoïdes")
    plt.title(
        f"{name} | k={k} | sil_strat={sil_strat:.3f} | sil={sil_mean:.3f}±{sil_std:.3f} | "
        f"DB={db:.3f} | ARI={ari_mean:.3f}"
    )
    plt.xlabel("Longitude (ontrack)")
    plt.ylabel("Latitude (ontrack)")
    plt.legend()
    plt.tight_layout()
    plt.savefig(fig_path, dpi=150)
    plt.close()
    t_plot = time.time() - t5

    t_total = time.time() - t0

    summary = {
        "test": name,
        "k": int(k),
        "n": int(n),
        "n_features": int(len(feat_cols)),
        "silhouette_strat": float(sil_strat),
        "silhouette_boot_mean": float(sil_mean),
        "silhouette_boot_std": float(sil_std),
        "davies_bouldin": float(db),
        "calinski_harabasz": float(ch),
        "inertia": float(inertia),
        "inertia_per_point": float(inertia / max(n, 1)),
        "cluster_min": min_c,
        "cluster_max": max_c,
        "cluster_ratio_max_min": ratio_max_min,
        "stability_ari_mean": float(ari_mean),
        "stability_ari_std": float(ari_std),
        "t_scale_s": round(t_scale, 2),
        "t_fit_s": round(t_fit, 2),
        "t_metrics_s": round(t_metrics, 2),
        "t_medoid_s": round(t_med, 2),
        "t_plot_s": round(t_plot, 2),
        "t_total_s": round(t_total, 2),
        "figure": fig_path.name,
        "medoids_csv": medoids_path.name
    }
    return summary

In [ ]:
# TESTS for k=3..6

# Dossiers sortie
REPORTS = Path("data/processed/analyse_exploratoire/clustering_reports")
MED = REPORTS / "medoids"
fc.create_folder(REPORTS)
fc.create_folder(MED)

# df existe contient:
# - lon_ontrack/lat_ontrack
# - meteo_cols
# - A00..A63 (BANDS)
# - PCA10_COLS / PCA15_COLS / PCA32_COLS

# Définition des sets de features
BANDS = [f"A{i:02d}" for i in range(64)]

TESTS = [
    ("meteo", meteo_cols),
    ("AE_raw", BANDS),
    ("meteo_plus_AEraw", meteo_cols + BANDS),
    ("AE_pca10", PCA10_COLS),
    ("AE_pca15", PCA15_COLS),
    ("AE_pca32", PCA32_COLS),
    ("meteo_plus_pca10", meteo_cols + PCA10_COLS),
    ("meteo_plus_pca15", meteo_cols + PCA15_COLS),
    ("meteo_plus_pca32", meteo_cols + PCA32_COLS),
]

# Loop k
for k in range(3, 7):
    print(f"\n==================== RUN k={k} ====================")
    results = []

    for name, cols in TESTS:
        print(f"[RUN] k={k} -> {name} | n_features={len(cols)}")
        res = kmeans_clusters_medoids_save(
            df_in=df,
            feat_cols=cols,
            name=name,
            k=k,
            out_dir_fig=CLUST,
            out_dir_medoids=MED,
            lon_col="lon_ontrack",
            lat_col="lat_ontrack",
            sample_sil_per_cluster=3000,
            boot_per_cluster=2000,
            boot_n=10,
            stability_runs=3,
            plot_max=120000,
            random_state=42,
            n_init=10
        )
        if res is not None:
            results.append(res)

    # Export report CSV for this k
    rep = pd.DataFrame(results).sort_values(
        by=["silhouette_boot_mean", "stability_ari_mean"],
        ascending=[False, False]
    )

    out_rep = REPORTS / f"clustering_tests_k{k}.csv"
    rep.to_csv(out_rep, index=False)
    print(f"[OK] Report écrit: {out_rep} | rows={len(rep)}")